## 1. 核心思想

这是一种 **"以查询为中心"** 的策略。利用 LLM 根据用户 Query 先生成一个"虚构的答案"（假设文档）。虽然这个答案内容可能包含幻觉，但其**语义模式**和**向量空间分布**与真实答案非常接近。然后用这个"假设文档"的向量去检索真实的文档块。

## 2. 核心逻辑伪代码

In [ ]:
def hyde_query_transform(user_query):
    """
    HyDE 核心逻辑：
    1. 输入：用户的问题
    2. 转化：让 LLM 写一篇“假文档”来回答这个问题
    3. 输出：这个“假文档”的向量
    """

    # Step 1: 生成假设文档 (Hypothetical Document)
    # 提示词很关键，要引导 LLM 生成类似知识库文档风格的文字
    prompt = f"""
    请为以下问题写一段科学、严谨的回答段落。
    不需要保证事实完全正确，但请使用专业的术语和语体。

    问题: {user_query}
    回答:
    """

    # 这里生成的 text 可能包含幻觉，但在向量空间里，
    # 它的位置会非常靠近真实的正确答案（因为包含相同的术语和语义结构）
    hypothetical_doc = llm.generate(prompt)

    # Step 2: 将“假文档”向量化
    # 注意：这里 encode 的是 hypothetical_doc，而不是 user_query
    search_vector = embedding_model.encode(hypothetical_doc)

    return search_vector, hypothetical_doc

## 3. 完整实现（Milvus + OpenAI）
我们将对比展示 HyDE 的工作流。

建库：构建一个标准的、未经优化的 Milvus 索引（模拟存量系统）。

HyDE 检索：用户提问 -> 生成假答案 -> 向量化 -> 检索真文档。

### 3.1 配置

In [1]:
import os
from pymilvus import MilvusClient
from openai import OpenAI

# --- 配置部分 ---
os.environ["OPENAI_API_KEY"] = "sk-OdqRypFlfJLKYvLmV6GsG9j0u6CRFBYKErn4xV1Wm0R3q0y9"  # 替换你的 Key
os.environ["OPENAI_BASE_URL"] = "https://api.openai-proxy.org/v1"
client = OpenAI()
milvus_client = MilvusClient(uri="http://111.228.53.183:19530")

COLLECTION_NAME = "standard_rag_index"
DIMENSION = 1536

### 3.2 核心函数封装

In [2]:
def get_embedding(text):
    """获取文本向量"""
    response = client.embeddings.create(
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

def generate_hypothetical_document(query):
    """
    HyDE 步骤1：生成假设性答案
    """
    prompt = f"""
    请扮演一个专业的各种技术专家。
    针对用户的问题，请写一段详细的、包含技术细节的回答段落。
    你的回答应该看起来像是一篇技术文档的片段。

    用户问题: "{query}"

    请直接输出回答内容。
    """

    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )
    return response.choices[0].message.content

### 3.3 准备milvus集合

In [3]:
if milvus_client.has_collection(COLLECTION_NAME):
    milvus_client.drop_collection(COLLECTION_NAME)

milvus_client.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=DIMENSION,
    auto_id=True,
    enable_dynamic_field=True
)

### 3.4 构建索引

In [4]:
documents = [
    # 文档 1: 关于索引类型
    "Milvus 支持多种向量索引类型。IVF_FLAT 是基于量化的索引，适合数据量适中的场景。HNSW（Hierarchical Navigable Small World）是一种基于图的索引，提供极高的查询速度和召回率，但在构建时消耗较多内存。ANNOY 是另一种基于树的近似最近邻搜索索引。",
    # 文档 2: 关于数据一致性
    "Milvus 支持四种一致性级别：Strong（强一致性）、Bounded（有界一致性）、Session（会话一致性）和 Eventually（最终一致性）。默认级别为 Bounded，这在搜索性能和数据新鲜度之间取得了平衡。",
    # 文档 3: 关于架构
    "Milvus 采用存储与计算分离的云原生架构。它包括 Access Layer（接入层）、Coordinator Service（协调服务）、Worker Node（工作节点）和 Storage（存储层）。这种设计使得系统具有弹性和高可用性。"
]

print("正在构建标准索引...")
data = []
for doc in documents:
    vec = get_embedding(doc)
    data.append({"vector": vec, "text": doc})
print("索引构建完成 (标准模式)。")

正在构建标准索引...
索引构建完成 (标准模式)。



### 3.5 插入milvus

In [ ]:
insert_result=milvus_client.insert(collection_name=COLLECTION_NAME, data=data)
print(f"\n成功插入 {insert_result['insert_count']} 条数据 (问题-文档映射)。\n")

## 3.6. 检索
- 定义检索函数

In [6]:
def hyde_search(user_query):
    print(f"原始用户查询: {user_query}")

    # 1. 生成假设文档 (The "Fake" Answer)
    print(">> 正在生成假设文档 (LLM Hallucinating)...")
    hypo_doc = generate_hypothetical_document(user_query)
    print(f"\n[生成的假设文档]: \n{hypo_doc}\n")

    # 2. 将假设文档向量化
    # 关键点：用假文档的向量去撞库
    hypo_vector = get_embedding(hypo_doc)

    # 3. 执行检索
    search_res = milvus_client.search(
        collection_name=COLLECTION_NAME,
        data=[hypo_vector], # 使用假文档向量
        limit=1,
        output_fields=["text"]
    )

    if not search_res:
        return "未找到相关文档。"

    retrieved_doc = search_res[0][0]["entity"]["text"]
    distance = search_res[0][0]["distance"]

    print(f">> 检索命中真实文档 (相似度: {distance:.4f})")
    print(f"[真实文档]: {retrieved_doc}\n")

    # 4. 最终答案生成 (使用真实文档)
    final_prompt = f"""
    基于以下真实文档回答用户问题。不要使用之前的假设文档内容。

    真实文档: {retrieved_doc}
    用户问题: {user_query}
    """

    final_answer = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": final_prompt}]
    )

    return final_answer.choices[0].message.content

## 3.7 检索
- 测试

In [7]:
# 场景：用户的问题很短，且可能没有包含具体的索引名称
# 普通检索直接搜 "Milvus 索引推荐" 可能会因为关键词匹配度不够而得分较低
# 但 HyDE 生成的假文档会包含 "IVF", "HNSW" 等词汇，从而拉回文档 1
query = "Milvus 有什么图索引吗？"

final_result = hyde_search(query)

print(f"最终 RAG 回答:\n{final_result}")

原始用户查询: Milvus 有什么图索引吗？
>> 正在生成假设文档 (LLM Hallucinating)...

[生成的假设文档]: 
Milvus 是一个开源的向量搜索引擎，它确实提供了多种图索引算法来帮助在大量数据中进行高效的向量搜索。这些图索引包括 HNSW (Hierarchical Navigable Small World) 和 IVF (Inverted File)。

HNSW 是一种基于图的索引方法，它通过创建一个分层的图结构，使得搜索路径可以在高层图中快速远距离地移动，并在底层图中进行精细搜索。这种方法允许进行相对高效的近似最近邻搜索。

IVF 是一种基于聚类的方法，它首先将向量空间分为多个簇，然后在每个簇内部建立索引。搜索时，只需要在目标向量所属簇以及其邻近的簇中进行搜索，大大提高了搜索效率。

除此之外，Milvus 还支持一些其他的索引方法，如 Flat、 IVF_FLAT、 IVF_SQ8、 IVF_PQ 等，它们适用于不同的应用场景和需求。在选择合适的索引方法时，需要考虑数据量、数据维度、查询效率、准确率等多个因素。

>> 检索命中真实文档 (相似度: 0.8384)
[真实文档]: Milvus 支持多种向量索引类型。IVF_FLAT 是基于量化的索引，适合数据量适中的场景。HNSW（Hierarchical Navigable Small World）是一种基于图的索引，提供极高的查询速度和召回率，但在构建时消耗较多内存。ANNOY 是另一种基于树的近似最近邻搜索索引。

最终 RAG 回答:
Milvus 支持的图索引类型是 HNSW（Hierarchical Navigable Small World）。这种索引提供了极高的查询速度和召回率，但在构建时会消耗较多的内存。


## 4. 适用场景

- 适用：

    - 零样本 (Zero-shot)：你不知道用户会怎么问，且你无法去微调 Embedding 模型。

    - 查询简短：用户输入 "耗电" 这种词，HyDE 可以将其扩充为 "电子设备电池耗电量及续航时间相关的技术参数..."，从而匹配到手册。
- 不适用：

    - 事实查证：如果查询是 "张三的工号是多少？"，LLM 编造一个 "张三工号 1234" 是没有意义的，因为 "1234" 这个向量无法匹配到真实的 "9527"。HyDE 适合概念性、描述性的检索，不适合精确的实体/数值检索